In [ ]:
import sys, os, json
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
RESP = sys.argv[1] if len(sys.argv) > 1 else r"C:\Users\aiala\OneDrive - Mondragon Unibertsitatea\BDA3\TENDENCIAS\Reto 09\Engagement Survey_BDATA3 26_27 (respuestas).xlsx"
ATTR = sys.argv[2] if len(sys.argv) > 2 else r"C:\Users\aiala\OneDrive - Mondragon Unibertsitatea\BDA3\TENDENCIAS\Reto 09\Employee Survey BData3_atributes_ex3 (2).xlsx"
OUT  = sys.argv[3] if len(sys.argv) > 3 else "."
os.makedirs(f"{OUT}/charts", exist_ok=True)

NAVY, TEAL, FAV, NEU, UNF, GREY = "#1B3A4B", "#0A8F99", "#3CC79A", "#FFC83D", "#FF8F80", "#DDE3E5"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
resp = pd.read_excel(r"C:\Users\aiala\OneDrive - Mondragon Unibertsitatea\BDA3\TENDENCIAS\Reto 09\Engagement Survey_BDATA3 26_27 (respuestas).xlsx")                                   # 1 fila = 1 persona, 1 columna = 1 pregunta
mp = pd.read_excel(ATTR, header=None).iloc[2:38, [1, 2, 4, 5]]  # mapeo pregunta -> feature -> atributo
mp.columns = ["n", "pregunta", "feature", "atributo"]
mp["n"] = mp["n"].astype(int)
mp["feature"] = mp["feature"].str.strip().replace({"Feedback Frecuency": "Feedback Frequency", "Role & Task": "Role & Tasks"})
mp["atributo"] = mp["atributo"].str.strip()
mp = mp.set_index("n")

norm = lambda s: " ".join(str(s).split()).lower().replace(" ?", "?")
q2n = {norm(q): n for n, q in mp["pregunta"].items()}
OPEN = {9, 10, 31} #preguntas abiertas
num, txt = {}, {}
for c in resp.columns[1:]:
    n = q2n[norm(c)]                                         
    if n in OPEN: txt[n] = resp[c]
    else:
        s = pd.to_numeric(resp[c], errors="coerce")
        num[n] = s.where(s.between(1, 5)) # 0 / vacio = fuera de escala: no respuesta
D = pd.DataFrame(num)
n_excl = int(sum(pd.to_numeric(resp[c], errors="coerce").between(1, 5).eq(False).sum() for c in resp.columns[1:] if q2n[norm(c)] not in OPEN))
print(f"Respuestas: {len(D)} | ítems Likert: {D.shape[1]} | celdas excluidas (0/vacío): {n_excl}")


Respuestas: 46 | ítems Likert: 33 | celdas excluidas (0/vacío): 10


In [ ]:
def stats(s):
    s = s.dropna()
    return dict(n=len(s), media=s.mean(), score=s.mean() * 2,          # score 0-10 = media(1-5) x 2
                fav=(s >= 4).mean() * 100, neu=(s == 3).mean() * 100, unf=(s <= 2).mean() * 100)

feat_items = {}
for n in D.columns: feat_items.setdefault((mp.loc[n, "atributo"], mp.loc[n, "feature"]), []).append(n)
F = pd.DataFrame([{**stats(pd.concat([D[i] for i in it])), "atributo": a, "feature": f, "items": it} for (a, f), it in feat_items.items()])
A = pd.DataFrame([{**stats(pd.concat([D[i] for i in its])), "atributo": a, "items": its, "n_features": len(g)}
                  for a, g in F.groupby("atributo", sort=False) for its in [[i for l in g["items"] for i in l]]])
ENG = A["score"].mean()                                                # Engagement score = media de atributos
A["estado"] = np.where(A.fav >= 80, "Fortaleza", np.where(A.fav < 60, "Oportunidad", "Estable"))
F["estado"] = np.where(F.fav >= 80, "Fortaleza", np.where(F.fav < 60, "Oportunidad", "Estable"))
I = pd.DataFrame({n: stats(D[n]) for n in D}).T
I["feature"] = [mp.loc[n, "feature"] for n in I.index]; I["pregunta"] = [mp.loc[n, "pregunta"] for n in I.index]
critical = I[(I.fav < 60) | (I.unf >= 20)].sort_values("fav")        # ítems críticos
print(f"ENGAGEMENT SCORE = {ENG:.2f}")
print(A[["atributo", "score", "fav", "neu", "unf", "estado"]].round(2).to_string(index=False))

ENGAGEMENT SCORE = 7.70
              atributo  score   fav   neu   unf      estado
RECOGNITION & FEEDBACK   6.78 52.22 32.22 15.56 Oportunidad
            WELL-BEING   7.65 67.03 27.47  5.49     Estable
       PERSONAL GROWTH   7.59 68.28 22.03  9.69     Estable
               CULTURE   7.82 72.79 19.12  8.09     Estable
    MANAGEMENT RAPPORT   7.77 71.04 24.04  4.92     Estable
           EMPOWERMENT   7.76 73.55 18.48  7.97     Estable
          SATISFACTION   7.72 75.18 19.71  5.11     Estable
          PEER RAPPORT   8.23 80.43 14.13  5.43   Fortaleza
              ADVOCACY   7.99 77.17 16.85  5.98     Estable


#### Drivers: correlacion de cada atributo con el indice de permanencia (me veo en 1 año, recomendaría, elegiría lo mismo)

In [ ]:
loy = D[[22, 30, 32]].mean(axis=1)
drv = {}
for _, r in A.iterrows():
    its = [i for i in r["items"] if i not in (22, 30, 32)]
    if its: drv[r["atributo"]] = D[its].mean(axis=1).corr(loy)
drv = pd.Series(drv).sort_values()

#### Temas de la pregunta abierta 10 (¿qué te impide tener más éxito?): codificación manual por número de fila (1-46)

In [ ]:
THEMES = {
 "Carga de trabajo / falta de tiempo": [1,3,7,13,18,23,24,26,32,35,37,41],
 "Falta de conocimientos / experiencia": [6,8,9,10,22,25,27,29,40],
 "Diseño del reto (límites, repetición, objetivos)": [5,11,14,15,28,34,42,45],
 "Compañeros (alineación, reparto, competitividad)": [4,16,30,36,44],
 "Actitud personal": [2,33,38,46],
 "Nada / sin respuesta": [12,17,19,20,21,31,39,43]}
assert sorted(sum(THEMES.values(), [])) == list(range(1, len(D) + 1)), "cada respuesta debe tener 1 tema"
T = pd.Series({k: len(v) for k, v in THEMES.items()})

## 3. GRÁFICOS

In [ ]:
def save(fig, name): fig.savefig(f"{OUT}/charts/{name}.png", dpi=170, bbox_inches="tight", facecolor="white"); plt.close(fig)

###  3.1 score por atributo

In [ ]:
a = A.sort_values("score")
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(a.atributo.str.title(), a.score, color=[TEAL if s >= 8 else NAVY for s in a.score], height=.62)
for y, (s, f) in enumerate(zip(a.score, a.fav)): ax.text(s + .08, y, f"{s:.2f}  ({f:.0f}% fav.)", va="center", fontsize=9)
ax.axvline(ENG, color=UNF, ls="--", lw=1.5); ax.text(ENG, len(a) - .45, f" Engagement {ENG:.2f}", color="#d9534f", fontsize=9)
ax.axvline(8, color="#999", ls=":", lw=1); ax.set_xlim(0, 10.8); ax.set_xlabel("Score (0-10)")
ax.set_title("Score de cada atributo", loc="left", fontweight="bold", color=NAVY); save(fig, "01_atributos")

### 3.2 score detail por feature (barras apiladas 100%), por grupos de atributos

In [ ]:
def detail(attrs, name, title):
    rows = [(a_, F[F.atributo == a_].sort_values("score", ascending=False)) for a_ in attrs]
    nrow = sum(len(g) + 1 for _, g in rows)
    fig, ax = plt.subplots(figsize=(9, 0.36 * nrow + .8)); y = nrow
    for a_, g in rows:
        y -= 1; ax.text(0, y, f"{a_.title()}  -  {A.set_index('atributo').loc[a_, 'score']:.2f}", fontweight="bold", color=TEAL, fontsize=10, va="center")
        for r in g.itertuples():
            y -= 1; left = 0
            for v, c in ((r.unf, UNF), (r.neu, NEU), (r.fav, FAV)):
                ax.barh(y, v, left=left, color=c, height=.7)
                if v >= 9: ax.text(left + v / 2, y, f"{v:.0f}%", ha="center", va="center", fontsize=8)
                left += v
            ax.text(-2, y, r.feature, ha="right", va="center", fontsize=9); ax.text(102, y, f"{r.score:.1f}", va="center", fontsize=9, fontweight="bold")
    ax.set_xlim(0, 108); ax.set_ylim(-.8, nrow); ax.axis("off")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in (UNF, NEU, FAV)], labels=["Desfavorable (1-2)", "Neutral (3)", "Favorable (4-5)"],
              loc="lower center", bbox_to_anchor=(.45, -.04 if nrow > 12 else -.12), ncol=3, frameon=False, fontsize=9)
    ax.set_title(title, loc="left", fontweight="bold", color=NAVY); save(fig, name)
detail(["EMPOWERMENT", "PERSONAL GROWTH", "PEER RAPPORT", "SATISFACTION"], "02_features_1", "Score detail por feature (1/2)")
detail(["WELL-BEING", "RECOGNITION & FEEDBACK", "CULTURE", "ADVOCACY", "MANAGEMENT RAPPORT"], "03_features_2", "Score detail por feature (2/2)")

### 3.3 drivers y 3.4 temas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4)); ax.barh(drv.index.str.title(), drv.values, color=[TEAL if v > .29 else GREY for v in drv.values], height=.6)
for y, v in enumerate(drv.values): ax.text(v + .01, y, f"{v:.2f}", va="center", fontsize=9)
ax.axvline(.29, color="#999", ls=":"); ax.set_xlabel("Correlación con la intención de permanencia (|r|>0.29 = significativa, n=46)")
ax.set_title("¿Qué atributos se asocian con querer quedarse?", loc="left", fontweight="bold", color=NAVY); save(fig, "04_drivers")
t = T.sort_values(); fig, ax = plt.subplots(figsize=(8, 3.8)); ax.barh(t.index, t.values, color=[NAVY if v == t.max() else TEAL for v in t.values], height=.6)
for y, v in enumerate(t.values): ax.text(v + .2, y, f"{v} ({v / len(D) * 100:.0f}%)", va="center", fontsize=9)
ax.set_xlim(0, 15); ax.set_title("¿Qué te impide tener más éxito? (respuestas abiertas)", loc="left", fontweight="bold", color=NAVY); save(fig, "05_temas")

## 4. EXPORTAR

In [ ]:
dump = lambda df: df.assign(items=df["items"].astype(str)) if "items" in df else df
with pd.ExcelWriter(f"{OUT}/Resultados_Engagement_BDATA3.xlsx") as w:
    pd.DataFrame({"Métrica": ["Respuestas", "Engagement score"], "Valor": [len(D), round(ENG, 2)]}).to_excel(w, sheet_name="Resumen", index=False)
    dump(A).round(2).to_excel(w, sheet_name="Score_Atributos", index=False); dump(F).round(2).to_excel(w, sheet_name="Score_Features", index=False)
    I.round(2).to_excel(w, sheet_name="Score_Items"); D.to_excel(w, sheet_name="Datos_limpios", index_label="respuesta")
    drv.round(3).rename("r").to_frame().to_excel(w, sheet_name="Drivers"); T.rename("n").to_frame().to_excel(w, sheet_name="Temas_abiertas")
json.dump(dict(n=len(D), eng=ENG, excl=n_excl, attrs=A.drop(columns="items").to_dict("records"), feats=F.drop(columns="items").to_dict("records"),
               critical=critical[["feature", "pregunta", "fav", "unf"]].to_dict("records"), drivers=drv.to_dict(), themes=T.to_dict()),
          open(f"{OUT}/results.json", "w"), ensure_ascii=False, default=float)
print("OK: charts/, Resultados_Engagement_BDATA3.xlsx, results.json")

OK -> charts/, Resultados_Engagement_BDATA3.xlsx, results.json
